# 03 · Reinforcement learning with verifiable rewards (GRPO-style)

Simplified from tinker-cookbook's `recipes/rl_loop.py`.

The big change: **the tokens now come from the model itself.** Each step:
1. **Sample** G attempts per problem from the current policy
2. **Score** each one (is the `\boxed{}` answer correct, within 256 tokens? 1 : 0)
3. **Advantage** = reward − mean reward of its group (better than siblings → reinforce)
4. **Train** with `importance_sampling` loss: like SFT, but each sampled token is
   weighted by its advantage (and can be negative!)

In [ ]:
from common import *

# ── LIVE (part 2): the same notebook on your own GPUs ──────────────────────
BASE_URL = None  # None = Tinker's cloud. SkyRL on this node: "http://localhost:8000"
connect(BASE_URL)

Backend: SkyRL at http://localhost:8000


In [7]:
train = load_gsm8k("train").shuffle(seed=1)
BATCH_SIZE, GROUP_SIZE, STEPS, LR = 16, 8, 20, 4e-5  # 16 problems x 8 attempts = 128 rollouts/step
if BASE_URL:
    STEPS = 5  # shorter live run on our 4 GPUs (~35 s/step)
sampling_params = types.SamplingParams(max_tokens=MAX_TOKENS, temperature=1.0, stop=renderer.get_stop_sequences())

# Start from the SFT checkpoint if we have one (it already knows the format), else from base.
# On SkyRL there's no `sft` checkpoint (it lives on Tinker's cloud), so it starts from base.
START_FROM = load_checkpoint("sft")
print("starting from:", START_FROM or f"base {MODEL}")

starting from: base Qwen/Qwen3.5-4B


In [8]:
# ── LIVE: the reward is just a Python function ────────────────────────────
def reward_fn(response: str, answer: str) -> float:
    return gsm8k_reward(response, answer)  # 1.0 if \boxed{} matches, else 0.0

In [9]:
# ── LIVE: the RL loop ──────────────────────────────────────────────────────
training_client = new_training_client(START_FROM)

for step in range(STEPS):
    rows = train.select(range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE))

    # 1. Sample from the *current* policy (fresh weights every step = on-policy)
    policy = training_client.save_weights_and_get_sampling_client().result()
    prompts = [renderer.build_generation_prompt(gsm8k_convo(q)) for q in rows["question"]]
    futures = [policy.sample(p, num_samples=GROUP_SIZE, sampling_params=sampling_params) for p in prompts]

    batch, step_rewards = [], []
    for prompt, fut, answer in zip(prompts, futures, rows["answer"]):
        seqs = fut.result().sequences
        # 2. Score each attempt
        texts = [renderers.get_text_content(renderer.parse_response(s.tokens)[0]) for s in seqs]
        rewards = [reward_fn(t, answer) for t in texts]
        step_rewards += rewards
        # 3. Group-relative advantage
        mean_r = sum(rewards) / len(rewards)
        if all(r == mean_r for r in rewards):
            continue  # all right or all wrong -> no signal
        # 4. One datum per attempt; the sampler's logprobs enable the importance-sampling correction
        for s, r in zip(seqs, rewards):
            batch.append(make_rl_datum(prompt, s.tokens, s.logprobs, advantages=r - mean_r))

    if batch:
        training_client.forward_backward(batch, loss_fn="importance_sampling")
        training_client.optim_step(adam(LR)).result()
    print(f"step {step:2d}  mean reward = {sum(step_rewards) / len(step_rewards):.3f}  ({len(batch)} datums)")

step  0  mean reward = 0.727  (24 datums)
step  1  mean reward = 0.570  (104 datums)
step  2  mean reward = 0.672  (96 datums)
step  3  mean reward = 0.688  (56 datums)
step  4  mean reward = 0.812  (48 datums)


Session heartbeat failed for 120.05256406823173 seconds for session session_31528050. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed for 130.0572666139342 seconds for session session_31528050. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed for 140.06361366203055 seconds for session session_31528050. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed for 150.06709324009717 seconds for session session_31528050. Last exception: APIConnectionError: Connection error..
Your connection may be unreliable or Tinker is down. If this persists, the session will be terminated.
Session heartbeat failed 

In [10]:
sampling_client = training_client.save_weights_and_get_sampling_client().result()
eval_gsm8k(sampling_client, label="3. RL" + (" (from SFT)" if START_FROM else ""))
save_checkpoint(training_client, "rl")

GSM8K accuracy within 256 tokens [SkyRL · 3. RL]: 87.0% (87/100)   [3 ran out of tokens]
saved 'SkyRL · rl' -> tinker://model_54ebea01/weights/rl


'tinker://model_54ebea01/weights/rl'

**Takeaway:** compare to SFT. SFT taught *every token* of a reference answer.
RL gives one bit per attempt (right/wrong) smeared over hundreds of tokens: a very
sparse signal, but it learns from the model's *own* mistakes.

**Bridge to distillation:** what if we could keep the on-policy sampling, but
get a grade on *every token* instead of one per answer?

---
## Part 2 · The same notebook on your own GPUs (SkyRL)

SkyRL implements the Tinker API on FSDP/Megatron + vLLM. **Colocated** mode: training and vLLM
share the same 4 GPUs and swap between steps.

1. Start the SkyRL server (cell below). It's ready in ~10 s; the model loads on the first `create_model` call.
2. At the top, set `BASE_URL = "http://localhost:8000"`.
3. Re-run the notebook from the top. Nothing else changes.

To re-run the RL loop, re-run the server cell first: on the FSDP backend a SkyRL server holds one LoRA adapter, so each run needs a fresh server (~10 s; the model loads on the first step either way).

In [ ]:
%%bash
# ── LIVE: start the SkyRL Tinker server on this node ──────────────────────
cd ~/default/skyrl-webinar
# Restart if one is already up: on FSDP a server holds one LoRA adapter, so each run needs a fresh one.
pkill -f 'skyrl\.tinker\.(api|engine)' && sleep 15 && pkill -9 -f 'skyrl\.tinker\.(api|engine)'

setsid nohup uv run --extra tinker --extra fsdp -m skyrl.tinker.api \
  --base-model "Qwen/Qwen3.5-4B" --backend fsdp --port 8000 \
  --backend-config '{
    "trainer.placement.colocate_all": true,
    "trainer.placement.policy_num_gpus_per_node": 4,
    "generator.inference_engine.num_engines": 4,
    "trainer.policy.language_model_only": true,
    "trainer.ref.language_model_only": true,
    "generator.inference_engine.language_model_only": true,
    "trainer.remove_microbatch_padding": false
  }' > logs/server.log 2>&1 &

until grep -qE "Uvicorn running|Traceback|ERROR" logs/server.log; do sleep 1; done
grep -q "Uvicorn running" logs/server.log && echo "SkyRL Tinker server is up on http://localhost:8000" || tail -n 30 logs/server.log

Stop it afterwards with `pkill -f 'skyrl\.tinker\.(api|engine)'` (it holds the GPUs). Log: `~/default/skyrl-webinar/logs/server.log`.

(`language_model_only` skips Qwen3.5's vision tower; sample packing isn't supported yet for Qwen3.5 on FSDP.)